In [ ]:
import sys
from pathlib import Path
import os
import pandas as pd
from io import BytesIO

ROOT_DIR = Path("c:\\Users\\rafae\\OneDrive\\Documentos\\F1")
sys.path.append(str(ROOT_DIR / "pipeline"))
from utils.storage import AzureBlobStorage
import logging
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
SILVER_CONTAINER = "processed"
GOLD_CONTAINER = "modelready"

storage_silver = AzureBlobStorage(AZURE_CONNECTION_STRING, SILVER_CONTAINER)
storage_gold = AzureBlobStorage(AZURE_CONNECTION_STRING, GOLD_CONTAINER)

blob_client = storage_silver.container_client.get_blob_client("silver_data.parquet")
parquet_bytes = blob_client.download_blob().readall()
df = pd.read_parquet(BytesIO(parquet_bytes))

logging.info(f"Parquet carregado Shape: {df.shape}")
df.head()

2025-12-06 02:11:53,804 - INFO - Request URL: 'https://stf1analytics01.blob.core.windows.net/processed/silver_data.parquet'
Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.11.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '14523ca0-d262-11f0-a2e1-a71f4788a230'
    'Authorization': 'REDACTED'
No body was attached to the request
2025-12-06 02:11:53,869 - INFO - Response status: 206
Response headers:
    'Content-Length': '25321'
    'Content-Type': 'application/octet-stream'
    'Content-Range': 'REDACTED'
    'Last-Modified': 'Sat, 06 Dec 2025 05:17:18 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DE3486BA090B50"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '3b0ccd04-901e-000d-6e6f-66265a000000'
    'x-ms-client-request-id': '14523ca0-d262-11f0-a2e1-a71f47

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,rainfall_mean,track_temperature_mean,air_temperature_mean,humidity_mean,pressure_mean,full_name,name_acronym,team_name,country_code_y,year
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,0.0,23.652866,18.227389,48.821656,1017.185987,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,0.0,23.652866,18.227389,48.821656,1017.185987,Lando NORRIS,NOR,McLaren,GBR,2024
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,0.0,23.652866,18.227389,48.821656,1017.185987,Oscar PIASTRI,PIA,McLaren,AUS,2024
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,0.0,31.593151,25.528082,62.335616,1012.682877,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,0.0,31.593151,25.528082,62.335616,1012.682877,Oscar PIASTRI,PIA,McLaren,AUS,2024


In [2]:
df.columns

Index(['position', 'driver_number', 'number_of_laps', 'dnf', 'dns', 'dsq',
       'gap_to_leader', 'duration', 'session_key', 'points', 'location',
       'country_code_x', 'country_name', 'circuit_short_name', 'lap_number',
       'category', 'flag', 'day', 'wind_speed_mean', 'wind_direction_mean',
       'rainfall_mean', 'track_temperature_mean', 'air_temperature_mean',
       'humidity_mean', 'pressure_mean', 'full_name', 'name_acronym',
       'team_name', 'country_code_y', 'year'],
      dtype='object')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   position                136 non-null    float64
 1   driver_number           141 non-null    int64  
 2   number_of_laps          139 non-null    float64
 3   dnf                     141 non-null    bool   
 4   dns                     141 non-null    bool   
 5   dsq                     141 non-null    bool   
 6   gap_to_leader           141 non-null    object 
 7   duration                133 non-null    float64
 8   session_key             141 non-null    int64  
 9   points                  141 non-null    float64
 10  location                141 non-null    object 
 11  country_code_x          141 non-null    object 
 12  country_name            141 non-null    object 
 13  circuit_short_name      141 non-null    object 
 14  lap_number              9 non-null      fl

In [4]:
df["driver_avg_position"] = (
    df.groupby("driver_number")["position"].transform("mean"))

df["driver_std_position"] = (
    df.groupby("driver_number")["position"].transform("std"))

df

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,air_temperature_mean,humidity_mean,pressure_mean,full_name,name_acronym,team_name,country_code_y,year,driver_avg_position,driver_std_position
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,18.227389,48.821656,1017.185987,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024,3.044444,2.275917
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,18.227389,48.821656,1017.185987,Lando NORRIS,NOR,McLaren,GBR,2024,3.978261,4.429643
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,18.227389,48.821656,1017.185987,Oscar PIASTRI,PIA,McLaren,AUS,2024,4.044444,2.875884
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,25.528082,62.335616,1012.682877,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024,3.044444,2.275917
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,25.528082,62.335616,1012.682877,Oscar PIASTRI,PIA,McLaren,AUS,2024,4.044444,2.875884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,22.599329,69.046980,1016.095302,Oscar PIASTRI,PIA,McLaren,AUS,2025,4.044444,2.875884
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,22.599329,69.046980,1016.095302,Lando NORRIS,NOR,McLaren,GBR,2025,3.978261,4.429643
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,17.917419,78.374194,988.776774,Lando NORRIS,NOR,McLaren,GBR,2025,3.978261,4.429643
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,17.917419,78.374194,988.776774,Oscar PIASTRI,PIA,McLaren,AUS,2025,4.044444,2.875884


In [5]:
import numpy as np

if "team_name" in df.columns:
    df["team_avg_position"] = (
        df.groupby("team_name")["position"].transform("mean")
    )
    df["team_std_position"] = (
        df.groupby("team_name")["position"].transform("std")
    )
else:
    df["team_avg_position"] = np.nan
    df["team_std_position"] = np.nan

df

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,pressure_mean,full_name,name_acronym,team_name,country_code_y,year,driver_avg_position,driver_std_position,team_avg_position,team_std_position
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,1017.185987,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024,3.044444,2.275917,3.044444,2.275917
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,1017.185987,Lando NORRIS,NOR,McLaren,GBR,2024,3.978261,4.429643,4.010989,3.722289
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,1017.185987,Oscar PIASTRI,PIA,McLaren,AUS,2024,4.044444,2.875884,4.010989,3.722289
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,1012.682877,Max VERSTAPPEN,VER,Red Bull Racing,NED,2024,3.044444,2.275917,3.044444,2.275917
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,1012.682877,Oscar PIASTRI,PIA,McLaren,AUS,2024,4.044444,2.875884,4.010989,3.722289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,1016.095302,Oscar PIASTRI,PIA,McLaren,AUS,2025,4.044444,2.875884,4.010989,3.722289
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,1016.095302,Lando NORRIS,NOR,McLaren,GBR,2025,3.978261,4.429643,4.010989,3.722289
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,988.776774,Lando NORRIS,NOR,McLaren,GBR,2025,3.978261,4.429643,4.010989,3.722289
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,988.776774,Oscar PIASTRI,PIA,McLaren,AUS,2025,4.044444,2.875884,4.010989,3.722289


In [6]:
df['circuit_short_name'].unique()

array(['Sakhir', 'Jeddah', 'Melbourne', 'Suzuka', 'Shanghai', 'Miami',
       'Imola', 'Monte Carlo', 'Montreal', 'Catalunya', 'Spielberg',
       'Silverstone', 'Hungaroring', 'Spa-Francorchamps', 'Zandvoort',
       'Monza', 'Baku', 'Singapore', 'Austin', 'Mexico City',
       'Interlagos', 'Las Vegas', 'Lusail', 'Yas Marina Circuit'],
      dtype=object)

In [7]:
df["track_avg_position"] = (
        df.groupby("circuit_short_name")["position"].transform("mean"))

df["track_std_position"] = (
    df.groupby("circuit_short_name")["position"].transform("std"))

df["driver_track_avg"] = (
    df.groupby(["driver_number", "circuit_short_name"])["position"]
        .transform("mean"))

df

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,team_name,country_code_y,year,driver_avg_position,driver_std_position,team_avg_position,team_std_position,track_avg_position,track_std_position,driver_track_avg
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,Red Bull Racing,NED,2024,3.044444,2.275917,3.044444,2.275917,4.166667,2.926887,3.5
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,McLaren,GBR,2024,3.978261,4.429643,4.010989,3.722289,4.166667,2.926887,4.5
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,McLaren,AUS,2024,4.044444,2.875884,4.010989,3.722289,4.166667,2.926887,4.5
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,Red Bull Racing,NED,2024,3.044444,2.275917,3.044444,2.275917,3.333333,2.658320,1.5
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,McLaren,AUS,2024,4.044444,2.875884,4.010989,3.722289,3.333333,2.658320,2.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,McLaren,AUS,2025,4.044444,2.875884,4.010989,3.722289,3.500000,3.391165,2.5
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,McLaren,GBR,2025,3.978261,4.429643,4.010989,3.722289,3.500000,3.391165,7.0
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,McLaren,GBR,2025,3.978261,4.429643,4.010989,3.722289,2.833333,1.471960,2.0
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,McLaren,AUS,2025,4.044444,2.875884,4.010989,3.722289,2.833333,1.471960,3.0


In [8]:
df.columns

Index(['position', 'driver_number', 'number_of_laps', 'dnf', 'dns', 'dsq',
       'gap_to_leader', 'duration', 'session_key', 'points', 'location',
       'country_code_x', 'country_name', 'circuit_short_name', 'lap_number',
       'category', 'flag', 'day', 'wind_speed_mean', 'wind_direction_mean',
       'rainfall_mean', 'track_temperature_mean', 'air_temperature_mean',
       'humidity_mean', 'pressure_mean', 'full_name', 'name_acronym',
       'team_name', 'country_code_y', 'year', 'driver_avg_position',
       'driver_std_position', 'team_avg_position', 'team_std_position',
       'track_avg_position', 'track_std_position', 'driver_track_avg'],
      dtype='object')

In [9]:
df["year"] = df["day"].astype(str).str[:4].astype(int)

df["race_index"] = (
    df.groupby("year")["session_key"].rank(method="dense")
)

df["driver_form_last3"] = (
    df.sort_values("session_key")
      .groupby("driver_number")["position"]
      .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

df

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,year,driver_avg_position,driver_std_position,team_avg_position,team_std_position,track_avg_position,track_std_position,driver_track_avg,race_index,driver_form_last3
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,2024,3.044444,2.275917,3.044444,2.275917,4.166667,2.926887,3.5,1.0,1.000000
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,2024,3.978261,4.429643,4.010989,3.722289,4.166667,2.926887,4.5,1.0,6.000000
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,2024,4.044444,2.875884,4.010989,3.722289,4.166667,2.926887,4.5,1.0,8.000000
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,2024,3.044444,2.275917,3.044444,2.275917,3.333333,2.658320,1.5,2.0,1.000000
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,2024,4.044444,2.875884,4.010989,3.722289,3.333333,2.658320,2.5,2.0,6.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,2025,4.044444,2.875884,4.010989,3.722289,3.500000,3.391165,2.5,2.0,6.333333
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,2025,3.978261,4.429643,4.010989,3.722289,3.500000,3.391165,7.0,2.0,2.333333
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,2025,3.978261,4.429643,4.010989,3.722289,2.833333,1.471960,2.0,13.0,1.333333
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,2025,4.044444,2.875884,4.010989,3.722289,2.833333,1.471960,3.0,13.0,1.666667


In [10]:
climate_cols = [
    "wind_speed_mean", "wind_direction_mean", "rainfall_mean",
    "track_temperature_mean", "air_temperature_mean",
    "humidity_mean", "pressure_mean"
]

for col in climate_cols:
    if col in df.columns:
        df[f"{col}_zscore"] = (df[col] - df[col].mean()) / df[col].std()

In [11]:
df["target_final_position"] = df["position"]
df["target_champion_points"] = df.groupby("driver_number")["points"].transform("sum")

In [12]:
df

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,driver_form_last3,wind_speed_mean_zscore,wind_direction_mean_zscore,rainfall_mean_zscore,track_temperature_mean_zscore,air_temperature_mean_zscore,humidity_mean_zscore,pressure_mean_zscore,target_final_position,target_champion_points
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,1.000000,-1.260657,-1.773315,-0.372325,-1.231942,-1.052473,-0.317152,0.582254,1.0,763.0
1,6.0,4,57.0,False,False,False,48.458,5553.200,9472,8.0,...,6.000000,-1.260657,-1.773315,-0.372325,-1.231942,-1.052473,-0.317152,0.582254,6.0,723.0
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,8.000000,-1.260657,-1.773315,-0.372325,-1.231942,-1.052473,-0.317152,0.582254,8.0,628.0
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,1.000000,-0.570439,-1.054119,-0.372325,-0.398964,0.463110,0.480960,0.493560,1.0,763.0
4,4.0,81,50.0,False,False,False,32.007,4875.280,9480,12.0,...,6.000000,-0.570439,-1.054119,-0.372325,-0.398964,0.463110,0.480960,0.493560,4.0,628.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,6.333333,-0.639537,0.541900,-0.372325,-0.759077,-0.144883,0.877322,0.560771,2.0,628.0
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,2.333333,-0.639537,0.541900,-0.372325,-0.759077,-0.144883,0.877322,0.560771,4.0,723.0
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,1.333333,-0.329366,-0.873850,1.027935,-1.239722,-1.116821,1.428171,0.022706,1.0,723.0
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,1.666667,-0.329366,-0.873850,1.027935,-1.239722,-1.116821,1.428171,0.022706,2.0,628.0


In [13]:
def flatten_list_columns(df):
    for col in df.columns:
        if df[col].dtype == object:
            has_list = any(isinstance(val, list) for val in df[col].dropna())
            has_mixed = False
            
            if not has_list:
                types_set = set()
                for val in df[col].dropna():
                    types_set.add(type(val).__name__)
                has_mixed = len(types_set) > 1
            
            if has_list or has_mixed:
                logging.info(f"Convertendo coluna '{col}' (contém listas ou tipos mistos) para string")
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (list, dict)) else str(x) if pd.notna(x) else None
                )
                df[col] = df[col].astype(str)
    
    return df

In [ ]:
def save_gold(df, filename="gold_data.parquet"):
    df = flatten_list_columns(df)
    df.to_parquet(filename, engine='pyarrow', index=False)
    
    logging.info(f"Salvando dados processados em {filename}")

    logging.info(f"Fazendo upload para {GOLD_CONTAINER}/{filename}")
    with open(filename, "rb") as f:
        storage_gold.container_client.upload_blob(name=filename, data=f, overwrite=True)
    
    logging.info(f"Upload concluído com sucesso!")

save_gold(df, filename="gold_data.parquet")

2025-12-06 02:11:54,102 - INFO - Processando DataTypes antes de salvar...
2025-12-06 02:11:54,110 - INFO - Salvando dados processados em gold_data.parquet
2025-12-06 02:11:54,113 - INFO - Fazendo upload para modelready/gold_data.parquet
2025-12-06 02:11:54,123 - INFO - Request URL: 'https://stf1analytics01.blob.core.windows.net/modelready/gold_data.parquet'
Request method: 'PUT'
Request headers:
    'Content-Length': '41668'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.11.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '1482e9b3-d262-11f0-bfe8-a71f4788a230'
    'Authorization': 'REDACTED'
A body is sent with the request
2025-12-06 02:11:54,243 - INFO - Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Sat, 06 Dec 2025 05:1